# Decorators in Python

## Overview
Decorators are one of Python's most powerful metaprogramming tools. At their core, a decorator is just a **function that takes a function and returns a function** — but this simple idea unlocks a massive design pattern: separating *cross-cutting concerns* (logging, timing, auth, caching) from *business logic*.

Why does this matter? Without decorators, you'd copy-paste the same timing or logging code into every function. With decorators, you write it once and apply it with a single line.

## Table of Contents
1. Functions as First-Class Objects
2. The Basic Decorator Pattern
3. The `@` Syntax Sugar
4. `functools.wraps` — Why It's Critical
5. Decorators with Arguments (Factory Pattern)
6. Stacking Multiple Decorators
7. Class-Based Decorators
8. Practical Decorators: `@timer`, `@retry`, `@cache`, `@validate_args`
9. Built-in Decorators Preview
10. Common Mistakes
11. Exercises
12. Mini-Project: Web Request Simulator

## 1. Functions as First-Class Objects

In Python, functions are **first-class objects** — they can be:
- Assigned to variables
- Passed as arguments to other functions
- Returned from functions
- Stored in data structures

This is the foundational insight that makes decorators possible.

In [ ]:
# Functions are objects — they have attributes
def greet(name):
    return f"Hello, {name}!"

print(greet)            # <function greet at 0x...>
print(greet.__name__)   # greet
print(type(greet))      # <class 'function'>

# Assign to a variable
say_hello = greet
print(say_hello("Alice"))  # Hello, Alice!

# Pass as argument
def call_twice(func, value):
    func(value)
    func(value)

call_twice(print, "ping")

In [ ]:
# Functions can be RETURNED from other functions (closures)
def make_multiplier(factor):
    """Returns a new function that multiplies by factor."""
    def multiplier(x):
        return x * factor   # 'factor' is captured in the closure
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

# The closure captures 'factor' even after make_multiplier has returned
print(double.__closure__[0].cell_contents)  # 2

## 2. The Basic Decorator Pattern

A decorator is a function that:
1. Accepts a function as input
2. Defines an inner `wrapper` function that adds behavior
3. Returns the `wrapper`

The key insight: you are **replacing** the original function with a new one that has extra behavior baked in.

In [ ]:
# Building a decorator from scratch — no magic yet
def shout_decorator(func):
    """Makes any function's return value uppercase."""
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)  # call the original
        return result.upper()           # add behavior
    return wrapper

def say_something(text):
    return f"I said: {text}"

# Manually applying the decorator
say_something = shout_decorator(say_something)

print(say_something("hello"))  # I SAID: HELLO

# This line: say_something = shout_decorator(say_something)
# ...is EXACTLY what the @ syntax does.

## 3. The `@` Syntax Sugar

The `@decorator` syntax is just a cleaner way to write `func = decorator(func)`. Python processes it at class/function definition time, not at call time.

In [ ]:
def shout_decorator(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

# The @ syntax — cleaner and more idiomatic
@shout_decorator
def greet(name):
    return f"hello, {name}"

print(greet("world"))  # HELLO, WORLD

# These two are identical:
# @shout_decorator
# def greet(name): ...
#
# def greet(name): ...
# greet = shout_decorator(greet)

## 4. `functools.wraps` — Why It's Critical

Without `functools.wraps`, your wrapper function **replaces the identity** of the original function. This breaks:
- `help()` and docstrings
- `__name__` inspection
- Debugging tools and stack traces
- Other decorators that depend on function metadata

Always use `@functools.wraps(func)` on your wrapper.

In [ ]:
import functools

# WITHOUT functools.wraps — identity is lost
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def important_function():
    """This docstring tells you something important."""
    pass

print(important_function.__name__)  # wrapper  <-- WRONG!
print(important_function.__doc__)   # None     <-- WRONG!

print("---")

# WITH functools.wraps — identity is preserved
def good_decorator(func):
    @functools.wraps(func)  # copies __name__, __doc__, __module__, etc.
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def important_function():
    """This docstring tells you something important."""
    pass

print(important_function.__name__)  # important_function  <-- correct!
print(important_function.__doc__)   # This docstring...   <-- correct!

## 5. Decorators with Arguments (Factory Pattern)

Sometimes you want `@repeat(3)` instead of `@repeat`. This requires **one more level of nesting** — a factory function that returns a decorator.

The pattern: `decorator_factory(arg)` → returns `decorator` → which takes `func` → which returns `wrapper`.

In [ ]:
import functools

# Decorator factory — outer function accepts arguments
def repeat(times):
    """Repeat a function call `times` times."""
    def decorator(func):         # this is the actual decorator
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result        # return last result
        return wrapper
    return decorator             # return the decorator

@repeat(times=3)
def say(message):
    print(message)

say("echo")
# echo
# echo
# echo

# What's happening:
# @repeat(times=3) calls repeat(3), which returns `decorator`
# Then `decorator` is applied to `say`, returning `wrapper`
# say now points to wrapper

## 6. Stacking Multiple Decorators

You can apply multiple decorators to a single function. The execution order is **bottom-up for application, top-down for execution**.

```
@A
@B
def f(): ...
```
This means: `f = A(B(f))`

In [ ]:
import functools

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return "<b>" + func(*args, **kwargs) + "</b>"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return "<i>" + func(*args, **kwargs) + "</i>"
    return wrapper

def underline(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return "<u>" + func(*args, **kwargs) + "</u>"
    return wrapper

@bold       # applied third (outermost)
@italic     # applied second
@underline  # applied first (innermost)
def text():
    return "Hello"

# Equivalent: text = bold(italic(underline(text)))
# When called: bold wraps italic wraps underline wraps original
print(text())  # <b><i><u>Hello</u></i></b>

## 7. Class-Based Decorators

A class can act as a decorator by implementing `__call__`. This is useful when you need to **maintain state** across calls (e.g., a call counter, a cache dict).

In [ ]:
import functools

class CallCounter:
    """Class-based decorator that counts how many times a function is called."""

    def __init__(self, func):
        functools.update_wrapper(self, func)  # preserve metadata (wraps equiv for classes)
        self.func = func
        self.count = 0   # state lives on the instance

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"[Call #{self.count}] Calling {self.func.__name__}")
        return self.func(*args, **kwargs)

@CallCounter
def add(a, b):
    return a + b

print(add(1, 2))
print(add(3, 4))
print(add(5, 6))
print(f"Total calls: {add.count}")

## 8. Practical Decorators

These are the decorators you'll actually use in real code.

In [ ]:
import functools
import time

# @timer — measures execution time
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__!r} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

result = slow_sum(10_000_000)
print(f"Result: {result}")

In [ ]:
import functools
import time
import random

# @retry — retries on exception
def retry(times=3, delay=0.5, exceptions=(Exception,)):
    """Retry decorator with configurable attempts, delay, and exception types."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    print(f"Attempt {attempt}/{times} failed: {e}")
                    if attempt < times:
                        time.sleep(delay)
            raise last_exception
        return wrapper
    return decorator

@retry(times=4, delay=0.1)
def flaky_operation():
    if random.random() < 0.7:   # 70% chance of failure
        raise ConnectionError("Network error!")
    return "Success!"

try:
    print(flaky_operation())
except ConnectionError:
    print("All attempts exhausted.")

In [ ]:
import functools

# @simple_cache — manual memoization (understand before using functools.lru_cache)
def simple_cache(func):
    """Cache results keyed by arguments. Note: only works for hashable args."""
    cache = {}   # dict lives in the closure

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Create a hashable key from arguments
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
            print(f"  [cache miss] computed {func.__name__}{args}")
        else:
            print(f"  [cache hit]  returned {func.__name__}{args}")
        return cache[key]

    wrapper.cache = cache       # expose cache for inspection
    wrapper.cache_clear = lambda: cache.clear()
    return wrapper

@simple_cache
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(10))
print(f"Cache size: {len(fib.cache)} entries")

# Python's built-in is better — use this in production
from functools import lru_cache

@lru_cache(maxsize=128)
def fib_builtin(n):
    if n <= 1:
        return n
    return fib_builtin(n - 1) + fib_builtin(n - 2)

print(fib_builtin(50))

In [ ]:
import functools

# @validate_args — enforce type constraints at runtime
def validate_types(**type_map):
    """
    Usage: @validate_types(name=str, age=int)
    Validates that named arguments match expected types.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            import inspect
            bound = inspect.signature(func).bind(*args, **kwargs)
            bound.apply_defaults()
            for param_name, expected_type in type_map.items():
                if param_name in bound.arguments:
                    value = bound.arguments[param_name]
                    if not isinstance(value, expected_type):
                        raise TypeError(
                            f"Argument '{param_name}' expected {expected_type.__name__}, "
                            f"got {type(value).__name__}"
                        )
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate_types(name=str, age=int)
def create_user(name, age):
    return {"name": name, "age": age}

print(create_user("Alice", 30))    # works fine

try:
    print(create_user("Bob", "30"))  # age is a string — should fail
except TypeError as e:
    print(f"TypeError: {e}")

## 9. Built-in Decorators Preview

Python provides three built-in decorators for class methods. These are covered in depth in notebook 02, but here's the essential distinction:
- `@staticmethod` — no implicit first argument, pure utility function
- `@classmethod` — receives the class (`cls`) as first argument
- `@property` — turns a method into an attribute-style accessor

In [ ]:
class Circle:
    PI = 3.14159

    def __init__(self, radius):
        self._radius = radius

    @property
    def area(self):           # call as circle.area, not circle.area()
        return self.PI * self._radius ** 2

    @classmethod
    def from_diameter(cls, diameter):   # alternative constructor
        return cls(diameter / 2)

    @staticmethod
    def is_valid_radius(r):   # no self, no cls — pure utility
        return r > 0

c = Circle(5)
print(c.area)                           # 78.53975
c2 = Circle.from_diameter(10)
print(c2._radius)                       # 5.0
print(Circle.is_valid_radius(-1))       # False

## 10. Common Mistakes

In [ ]:
# MISTAKE 1: Calling the function when applying the decorator
import functools

def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("Before")
        return func(*args, **kwargs)
    return wrapper

# WRONG: @my_decorator()  <-- calling it adds () which means
# my_decorator() returns None, then Python tries to call None(func) — TypeError

# CORRECT:
@my_decorator   # no parentheses
def hello():
    print("Hello!")

hello()

In [ ]:
# MISTAKE 2: Forgetting to return the result from wrapper
import functools

def broken_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        func(*args, **kwargs)   # missing return!
    return wrapper

@broken_decorator
def add(a, b):
    return a + b

result = add(2, 3)
print(result)   # None — the return value was swallowed!

# Fix: always use `return func(*args, **kwargs)` in your wrapper

In [ ]:
# MISTAKE 3: Decorator factory without arguments — confusing the two levels
import functools

# Suppose you want @log to work both with and without arguments:
# @log          (no args)
# @log(level=2) (with args)
# The trick: check if first argument is a callable

def log(_func=None, *, level=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(f"[L{level}] Calling {func.__name__}")
            return func(*args, **kwargs)
        return wrapper

    if _func is not None:
        return decorator(_func)  # used as @log (no args)
    return decorator             # used as @log(...)

@log
def fn1(): return "fn1"

@log(level=3)
def fn2(): return "fn2"

fn1()
fn2()

## 11. Exercises

**Exercise 1 (Easy):** Write a `@debug` decorator that prints the function name, arguments, and return value every time it's called.

**Exercise 2 (Medium):** Write a `@throttle(calls, period)` decorator that limits a function to at most `calls` invocations per `period` seconds. Raise a `RuntimeError` if the limit is exceeded.

**Exercise 3 (Hard):** Write a `@memoize_with_ttl(ttl_seconds)` decorator. Cached results expire after `ttl_seconds`. After expiry the function is called again and the cache is refreshed.

In [ ]:
# Exercise 1 Solution: @debug
import functools

def debug(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        args_repr = [repr(a) for a in args]
        kwargs_repr = [f"{k}={v!r}" for k, v in kwargs.items()]
        signature = ", ".join(args_repr + kwargs_repr)
        print(f"Calling {func.__name__}({signature})")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result!r}")
        return result
    return wrapper

@debug
def multiply(a, b):
    return a * b

multiply(4, b=5)

In [ ]:
# Exercise 2 Solution: @throttle
import functools
import time
from collections import deque

def throttle(calls, period):
    def decorator(func):
        call_times = deque()   # sliding window of timestamps

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            now = time.monotonic()
            # Remove timestamps older than the window
            while call_times and now - call_times[0] > period:
                call_times.popleft()
            if len(call_times) >= calls:
                raise RuntimeError(
                    f"{func.__name__} exceeded {calls} calls per {period}s"
                )
            call_times.append(now)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@throttle(calls=3, period=1.0)
def api_call(n):
    return f"response {n}"

for i in range(5):
    try:
        print(api_call(i))
    except RuntimeError as e:
        print(f"Throttled: {e}")

In [ ]:
# Exercise 3 Solution: @memoize_with_ttl
import functools
import time

def memoize_with_ttl(ttl_seconds):
    def decorator(func):
        cache = {}   # key -> (result, timestamp)

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            now = time.monotonic()
            if key in cache:
                result, ts = cache[key]
                if now - ts < ttl_seconds:
                    print(f"  [hit, age={(now-ts):.2f}s]")
                    return result
                else:
                    print(f"  [expired after {now-ts:.2f}s]")
            result = func(*args, **kwargs)
            cache[key] = (result, now)
            return result
        return wrapper
    return decorator

@memoize_with_ttl(ttl_seconds=0.5)
def expensive(n):
    print(f"  [computing expensive({n})]")
    return n ** 2

print(expensive(5))       # computes
print(expensive(5))       # cache hit
time.sleep(0.6)
print(expensive(5))       # expired, recomputes

## 12. Mini-Project: Web Request Simulator

We'll build a realistic simulation of decorated API calls. The goal: apply `@retry` and `@log_calls` decorators to a mock API function that randomly fails, demonstrating how decorators compose in real applications.

In [ ]:
import functools
import time
import random
import datetime

# --- Decorator 1: @retry ---
def retry(times=3, delay=0.5, exceptions=(Exception,)):
    """Retry the wrapped function up to `times` times on specified exceptions."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    print(f"  [retry] attempt {attempt}/{times} failed: {exc}")
                    if attempt < times:
                        time.sleep(delay)
                    else:
                        raise
        return wrapper
    return decorator


# --- Decorator 2: @log_calls ---
def log_calls(func):
    """Log every call with timestamp, arguments, result or exception."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        ts = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
        arg_str = ", ".join([repr(a) for a in args] +
                             [f"{k}={v!r}" for k, v in kwargs.items()])
        print(f"[{ts}] CALL  {func.__name__}({arg_str})")
        try:
            result = func(*args, **kwargs)
            print(f"[{ts}] OK    {func.__name__} -> {result!r}")
            return result
        except Exception as exc:
            print(f"[{ts}] ERROR {func.__name__} raised {type(exc).__name__}: {exc}")
            raise
    return wrapper


# --- Mock API function with random failures ---
class APIError(Exception):
    pass

FAILURE_RATE = 0.6   # 60% chance of failure

@log_calls                             # outer: executes first on call
@retry(times=3, delay=0.1, exceptions=(APIError,))  # inner: handles retries
def fetch_user(user_id: int) -> dict:
    """Simulates a network call to fetch a user by ID."""
    if random.random() < FAILURE_RATE:
        raise APIError(f"503 Service Unavailable for user {user_id}")
    return {"id": user_id, "name": f"User_{user_id}", "active": True}


# --- Demo ---
print("=" * 55)
print("Web Request Simulator")
print("=" * 55)

random.seed(42)

for uid in [101, 202, 303]:
    print(f"\nFetching user {uid}...")
    try:
        user = fetch_user(uid)
        print(f"Got: {user}")
    except APIError as e:
        print(f"All retries exhausted: {e}")